# Process website_alldata (clean pipeline)

This notebook builds **two different V2 outputs** on purpose:

- `website_full_matrix.csv` = V2 full table in **sequences.csv-style**:
  - columns: `ID`, `seqs`, then disease 0/1 columns
  - one row per V2 lncRNA ID (even if `seqs` is missing)
- `website_sequences_for_oop.csv` = extractor input only:
  - columns: `id`, `seq`
  - includes **only rows with non-empty sequences**

V1 output in this notebook:
- `sequences_for_oop.csv` is kept in V1 full style (`ID`, `seqs`, disease columns), sourced from `Data/raw/sequences.csv`.

Other outputs (`Data/output_data`):
- `ncrna_symbol_list.txt`
- `website_sequences.csv` (ID + seqs fetch cache)
- `sequence_fetch_report.csv` (status/details per V2 ID)
- `website_missing_sequence_ids.txt` (IDs still missing seq after fetch)
- `website_disease_matrix.csv`
- `dinuc_props.csv`
- `do_terms.csv`, `do_edges.csv`, `disease_terms_mapping.csv`


In [ ]:
from pathlib import Path
import re
import sys
import time
import requests
import pandas as pd

try:
    import obonet
except Exception:
    obonet = None


def find_project_root(marker_rel: Path = Path("Data/raw/website_alldata.csv")) -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / marker_rel).exists():
            return base
    raise FileNotFoundError(f"Could not locate project root containing {marker_rel}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mainfolder.utils.sequence_fetch import (
    detect_id_kind,
    fetch_sequence_by_id,
    has_usable_sequence_value,
    needs_manual_review,
    normalize_sequence_value,
)

RAW_WEBSITE = PROJECT_ROOT / "Data/raw/website_alldata.csv"
RAW_V1 = PROJECT_ROOT / "Data/raw/sequences.csv"
DO_OBO = PROJECT_ROOT / "Data/raw/HumanDO.obo"
OUT_DIR = PROJECT_ROOT / "Data/output_data"

SYMBOLS_TXT = OUT_DIR / "ncrna_symbol_list.txt"
WEBSITE_SEQS = OUT_DIR / "website_sequences.csv"
FETCH_REPORT = OUT_DIR / "sequence_fetch_report.csv"
WEBSITE_MISSING_IDS = OUT_DIR / "website_missing_sequence_ids.txt"
UNRESOLVED_IDS = OUT_DIR / "unresolved_ids.csv"
AMBIGUOUS_ALTERNATIVES = OUT_DIR / "ambiguous_symbol_alternatives.csv"

WEBSITE_DISEASE = OUT_DIR / "website_disease_matrix.csv"
WEBSITE_FULL = OUT_DIR / "website_full_matrix.csv"
WEBSITE_OOP = OUT_DIR / "website_sequences_for_oop.csv"

V1_OOP = OUT_DIR / "sequences_for_oop.csv"
V1_FETCH_REPORT = OUT_DIR / "sequence_fetch_report_v1.csv"
V1_UNRESOLVED_IDS = OUT_DIR / "unresolved_ids_v1.csv"
V1_AMBIGUOUS_ALTERNATIVES = OUT_DIR / "ambiguous_symbol_alternatives_v1.csv"

DINUC_PROPS = OUT_DIR / "dinuc_props.csv"
DO_TERMS = OUT_DIR / "do_terms.csv"
DO_EDGES = OUT_DIR / "do_edges.csv"
DO_MAP = OUT_DIR / "disease_terms_mapping.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_WEBSITE.exists():
    raise FileNotFoundError(f"Missing input: {RAW_WEBSITE}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Using website raw: {RAW_WEBSITE}")
print(f"Output dir: {OUT_DIR}")




In [ ]:
# 1) Load + filter to human lncRNA only (single source of truth)
raw = pd.read_csv(RAW_WEBSITE, dtype=str).fillna("")

required_cols = ["Species", "ncRNA Category", "ncRNA Symbol", "Disease Name"]
missing_cols = [c for c in required_cols if c not in raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {RAW_WEBSITE}: {missing_cols}")

flt = raw[
    raw["Species"].astype(str).str.strip().str.lower().eq("homo sapiens")
    & raw["ncRNA Category"].astype(str).str.strip().str.lower().eq("lncrna")
].copy()

flt["ncRNA Symbol"] = flt["ncRNA Symbol"].astype(str).str.strip()
flt["Disease Name"] = flt["Disease Name"].astype(str).str.strip()
flt = flt[(flt["ncRNA Symbol"] != "") & (flt["Disease Name"] != "")]

print("raw rows:", len(raw))
print("filtered rows (human + lncRNA):", len(flt))
print("unique symbols:", flt["ncRNA Symbol"].nunique())
print("unique diseases:", flt["Disease Name"].nunique())


In [ ]:
# 2) Symbol list for sequence fetching (from filtered website V2 set)
symbols = sorted(flt["ncRNA Symbol"].dropna().astype(str).str.strip().unique())
with open(SYMBOLS_TXT, "w") as f:
    for s in symbols:
        if s:
            f.write(s + "\n")
print(f"[saved] {SYMBOLS_TXT} ({len(symbols)} symbols)")


In [ ]:
# 3) Build website_sequences.csv (ID + seqs) for ALL filtered symbols
# Priority:
# 1) Reuse existing website_sequences.csv (resume)
# 2) Use direct sequence column if it exists in raw
# 3) Fetch unresolved IDs online with shared resolver rules

FORCE_REFETCH = False
ALLOW_ONLINE_FETCH = True
MAX_SYMBOLS = None      # set int for quick test, e.g. 200
SLEEP_SEC = 0.05
TIMEOUT = 10        # seconds per API request
PROGRESS_EVERY = 25 # print progress every N unresolved IDs

# target IDs we must attempt to resolve
all_symbols = symbols[:MAX_SYMBOLS] if MAX_SYMBOLS else symbols
target_ids = [s for s in all_symbols if isinstance(s, str) and s.strip()]
target_set = set(target_ids)

# existing cache from previous run
existing_df = pd.DataFrame(columns=["ID", "seqs"])
if WEBSITE_SEQS.exists() and not FORCE_REFETCH:
    existing_df = pd.read_csv(WEBSITE_SEQS, dtype=str).fillna("")
    if "ID" not in existing_df.columns or "seqs" not in existing_df.columns:
        existing_df = pd.DataFrame(columns=["ID", "seqs"])
    else:
        existing_df = existing_df[["ID", "seqs"]]
        existing_df["seqs"] = existing_df["seqs"].map(normalize_sequence_value)
        invalid_cached_n = int(existing_df["seqs"].eq("").sum())
        if invalid_cached_n:
            print(f"[cache] ignored {invalid_cached_n} invalid cached sequence values from {WEBSITE_SEQS.name}")

# direct extraction if raw has sequence-like column
possible_seq_cols = ["seqs", "seq", "sequence", "sequences", "cdna sequences", "cDNA sequences"]
possible_seq_cols_l = {x.lower() for x in possible_seq_cols}
seq_col = next((c for c in flt.columns if str(c).strip().lower() in possible_seq_cols_l), None)

direct_df = pd.DataFrame(columns=["ID", "seqs"])
if seq_col is not None:
    direct_df = (
        flt[["ncRNA Symbol", seq_col]]
        .rename(columns={"ncRNA Symbol": "ID", seq_col: "seqs"})
        .fillna("")
    )
    direct_df["seqs"] = direct_df["seqs"].map(normalize_sequence_value)

direct_ids = set(direct_df["ID"].astype(str).str.strip().tolist()) if not direct_df.empty else set()

# merge candidate sources, prefer rows with non-empty seqs
cand = pd.concat([existing_df, direct_df], ignore_index=True)
if not cand.empty:
    cand["ID"] = cand["ID"].astype(str).str.strip()
    cand["seqs"] = cand["seqs"].map(normalize_sequence_value)
    cand = cand[cand["ID"] != ""]
    cand = cand[cand["ID"].isin(target_set)]
    cand["_has_seq"] = cand["seqs"].map(has_usable_sequence_value)
    cand = cand.sort_values(["ID", "_has_seq"], ascending=[True, False]).drop_duplicates(subset=["ID"], keep="first")
else:
    cand = pd.DataFrame(columns=["ID", "seqs", "_has_seq"])

seq_map = {r.ID: normalize_sequence_value(r.seqs) for r in cand.itertuples()}
status_map = {}
detail_map = {}
type_map = {}
resolved_id_map = {}
alternative_rows = []

for rid in target_ids:
    seqv = normalize_sequence_value(seq_map.get(rid, ""))
    type_map[rid] = detect_id_kind(rid)
    if seqv:
        if seq_col is not None and rid in direct_ids:
            status_map[rid] = "from_raw"
            detail_map[rid] = f"column:{seq_col}"
        else:
            status_map[rid] = "cached"
            detail_map[rid] = "existing website_sequences.csv"
        resolved_id_map[rid] = rid if type_map[rid] in {"ensembl", "accession"} else ""
    else:
        status_map[rid] = "pending"
        detail_map[rid] = ""
        resolved_id_map[rid] = ""

unresolved = [rid for rid in target_ids if not has_usable_sequence_value(seq_map.get(rid, ""))]

if unresolved and ALLOW_ONLINE_FETCH:
    session = requests.Session()

    total_unresolved = len(unresolved)
    print(f"[fetch-config] timeout={TIMEOUT}s sleep={SLEEP_SEC}s progress_every={PROGRESS_EVERY}")
    print(f"[fetch] starting unresolved fetch for {total_unresolved} IDs")

    for i, rid in enumerate(unresolved, start=1):
        result = fetch_sequence_by_id(session, rid, timeout=TIMEOUT)
        type_map[rid] = result.id_type
        if result.sequence:
            seq_map[rid] = normalize_sequence_value(result.sequence)
        else:
            seq_map[rid] = ""
        status_map[rid] = result.status
        detail_map[rid] = result.detail
        resolved_id_map[rid] = result.resolved_id
        if result.alternative_ids:
            alternative_rows.append(result.alternatives_row())

        if i == 1 or i % PROGRESS_EVERY == 0 or i == total_unresolved or not result.sequence:
            print(
                f"[fetch] {i}/{total_unresolved} id={rid} kind={result.id_type} "
                f"status={status_map[rid]} detail={detail_map[rid]}"
            )

        time.sleep(SLEEP_SEC)
elif unresolved and not ALLOW_ONLINE_FETCH:
    for rid in unresolved:
        status_map[rid] = "fetch_disabled"
        detail_map[rid] = "ALLOW_ONLINE_FETCH=False"
        type_map[rid] = detect_id_kind(rid)
        resolved_id_map[rid] = ""

# write final website_sequences.csv with one row per target ID (even if seq missing)
rows = [{"ID": rid, "seqs": normalize_sequence_value(seq_map.get(rid, ""))} for rid in target_ids]
seqs_out = pd.DataFrame(rows)
seqs_out.to_csv(WEBSITE_SEQS, index=False)

report = pd.DataFrame(
    [
        {
            "ID": rid,
            "type": type_map.get(rid, detect_id_kind(rid)),
            "status": status_map.get(rid, "unknown"),
            "detail": detail_map.get(rid, ""),
            "resolved_id": resolved_id_map.get(rid, ""),
        }
        for rid in target_ids
    ]
)
report.to_csv(FETCH_REPORT, index=False)

missing_ids = [rid for rid in target_ids if not has_usable_sequence_value(seq_map.get(rid, ""))]
WEBSITE_MISSING_IDS.write_text("\n".join(missing_ids), encoding="utf-8")

manual_rows = []
for rid in target_ids:
    seqv = normalize_sequence_value(seq_map.get(rid, ""))
    if seqv:
        continue
    status = status_map.get(rid, "unknown")
    detail = detail_map.get(rid, "")
    id_type = type_map.get(rid, detect_id_kind(rid))
    resolved_id = resolved_id_map.get(rid, "")

    if status == "ambiguous_symbol":
        reason = detail or "symbol appears non-unique"
    elif status in {"unresolved", "fetch_disabled", "pending", "failed", "unknown"}:
        reason = detail or "unresolved after symbol/accession/ensembl checks"
    else:
        continue

    manual_rows.append(
        {
            "id": rid,
            "type": id_type,
            "status": status,
            "reason": reason,
            "resolved_id": resolved_id,
        }
    )

unresolved_df = pd.DataFrame(manual_rows, columns=["id", "type", "status", "reason", "resolved_id"])
if not unresolved_df.empty:
    unresolved_df = unresolved_df.drop_duplicates(subset=["id"], keep="first")
unresolved_df.to_csv(UNRESOLVED_IDS, index=False)

alternatives_df = pd.DataFrame(
    alternative_rows,
    columns=["query_id", "selected_id", "alternative_ids", "source"],
)
if not alternatives_df.empty:
    alternatives_df = alternatives_df.drop_duplicates(subset=["query_id"], keep="first")
alternatives_df.to_csv(AMBIGUOUS_ALTERNATIVES, index=False)

resolved_n = int(seqs_out["seqs"].map(has_usable_sequence_value).sum())
print(f"[saved] {WEBSITE_SEQS} shape={seqs_out.shape} (resolved={resolved_n}/{len(seqs_out)})")
print(f"[saved] {FETCH_REPORT} shape={report.shape}")
print(f"[saved] {WEBSITE_MISSING_IDS} missing_ids={len(missing_ids)}")
print(f"[saved] {UNRESOLVED_IDS} rows={len(unresolved_df)}")
print(f"[saved] {AMBIGUOUS_ALTERNATIVES} rows={len(alternatives_df)}")
if missing_ids:
    print("[info] missing IDs can be reviewed in website_missing_sequence_ids.txt, unresolved_ids.csv, and ambiguous_symbol_alternatives.csv")




In [ ]:
# 4) Build disease matrix from filtered data (ID + 0/1 disease columns)
flt2 = flt.copy()
flt2["present"] = 1

# one row per ID, one column per disease label (multi-label matrix)
disease_mat = (
    flt2.pivot_table(
        index="ncRNA Symbol",
        columns="Disease Name",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
    .rename(columns={"ncRNA Symbol": "ID"})
)

disease_mat["ID"] = disease_mat["ID"].astype(str).str.strip()
disease_mat = disease_mat[disease_mat["ID"] != ""]

# keep stable ordering for reproducibility
disease_cols = sorted([c for c in disease_mat.columns if c != "ID"])
disease_mat = disease_mat[["ID", *disease_cols]]

disease_mat.to_csv(WEBSITE_DISEASE, index=False)
print(f"[saved] {WEBSITE_DISEASE} shape={disease_mat.shape}")


In [ ]:
# 5) Merge sequence table + disease matrix into V2 full matrix (sequences.csv-style)
# Target structure: ID, seqs, <disease 0/1 columns>

seqs = None
if WEBSITE_SEQS.exists():
    seqs = pd.read_csv(WEBSITE_SEQS, dtype=str).fillna("")
    print(f"Using {WEBSITE_SEQS} shape={seqs.shape}")
elif WEBSITE_FULL.exists():
    full_prev = pd.read_csv(WEBSITE_FULL, dtype=str).fillna("")
    if {"ID", "seqs"}.issubset(full_prev.columns):
        seqs = full_prev[["ID", "seqs"]].drop_duplicates(subset=["ID"], keep="first")
        seqs.to_csv(WEBSITE_SEQS, index=False)
        print(f"[recovered] {WEBSITE_SEQS} from existing {WEBSITE_FULL} shape={seqs.shape}")

if seqs is not None:
    seqs["ID"] = seqs["ID"].astype(str).str.strip()
    seqs["seqs"] = seqs["seqs"].map(normalize_sequence_value)
    seqs = seqs[seqs["ID"] != ""]

    # one row per ID, prefer non-empty sequences when duplicates exist
    seqs["_has_seq"] = seqs["seqs"].map(has_usable_sequence_value)
    seqs = seqs.sort_values(["ID", "_has_seq"], ascending=[True, False]).drop_duplicates(subset=["ID"], keep="first")
    seqs = seqs[["ID", "seqs"]]

    # LEFT join keeps all disease IDs, even if sequence is still missing
    full = disease_mat.merge(seqs, on="ID", how="left")
    full["seqs"] = full["seqs"].map(normalize_sequence_value)

    disease_cols = [c for c in disease_mat.columns if c != "ID"]
    full = full[["ID", "seqs", *disease_cols]]
    full.to_csv(WEBSITE_FULL, index=False)

    missing_seq_n = int((~full["seqs"].map(has_usable_sequence_value)).sum())
    print(f"[saved] {WEBSITE_FULL} shape={full.shape} | missing seqs={missing_seq_n}")

    # website_sequences_for_oop.csv is intentionally the non-missing subset only
    oop = full.loc[full["seqs"].map(has_usable_sequence_value), ["ID", "seqs"]].rename(columns={"ID": "id", "seqs": "seq"})
    oop.to_csv(WEBSITE_OOP, index=False)
    print(f"[saved] {WEBSITE_OOP} shape={oop.shape}")
else:
    print(
        f"[skip] no sequence source found. Missing both {WEBSITE_SEQS} and recoverable {WEBSITE_FULL}.\n"
        "Run step 3 first to create website_sequences.csv."
    )


In [ ]:
# 6) V1: preserve full sequences.csv structure and fill missing seqs if needed
# Output path is kept for compatibility: Data/output_data/sequences_for_oop.csv

if RAW_V1.exists():
    v1 = pd.read_csv(RAW_V1, dtype=str)

    if "ID" not in v1.columns or "seqs" not in v1.columns:
        raise ValueError(f"{RAW_V1} must contain 'ID' and 'seqs' columns")

    v1["ID"] = v1["ID"].astype(str).str.strip()
    seq_clean = v1["seqs"].map(normalize_sequence_value)
    v1["seqs"] = seq_clean
    missing_mask = seq_clean.eq("")
    missing_ids = sorted(set(v1.loc[missing_mask, "ID"].tolist()) - {""})

    report_rows = []
    review_rows = []
    alternative_rows = []
    fetched = {}

    if missing_ids:
        session = requests.Session()
        for rid in missing_ids:
            result = fetch_sequence_by_id(session, rid, timeout=12)
            if result.alternative_ids:
                alternative_rows.append(result.alternatives_row())
            report_rows.append(result.report_row(id_column="ID"))
            if result.sequence:
                fetched[rid] = result.sequence
            elif needs_manual_review(result):
                review_rows.append(result.review_row())

        if fetched:
            fill_mask = missing_mask & v1["ID"].isin(fetched)
            v1.loc[fill_mask, "seqs"] = v1.loc[fill_mask, "ID"].map(fetched)
    v1["seqs"] = v1["seqs"].map(normalize_sequence_value)

    # Keep same structure/columns as raw sequences.csv (do not collapse to id/seq)
    v1.to_csv(V1_OOP, index=False)

    # Save fetch report only if fetch was needed
    if report_rows:
        pd.DataFrame(report_rows).to_csv(V1_FETCH_REPORT, index=False)
        print(f"[saved] {V1_FETCH_REPORT} rows={len(report_rows)}")

    v1_unresolved_df = pd.DataFrame(review_rows, columns=["id", "type", "status", "reason", "resolved_id"])
    if not v1_unresolved_df.empty:
        v1_unresolved_df = v1_unresolved_df.drop_duplicates(subset=["id"], keep="first")
    v1_unresolved_df.to_csv(V1_UNRESOLVED_IDS, index=False)
    print(f"[saved] {V1_UNRESOLVED_IDS} rows={len(v1_unresolved_df)}")

    v1_alternatives_df = pd.DataFrame(
        alternative_rows,
        columns=["query_id", "selected_id", "alternative_ids", "source"],
    )
    if not v1_alternatives_df.empty:
        v1_alternatives_df = v1_alternatives_df.drop_duplicates(subset=["query_id"], keep="first")
    v1_alternatives_df.to_csv(V1_AMBIGUOUS_ALTERNATIVES, index=False)
    print(f"[saved] {V1_AMBIGUOUS_ALTERNATIVES} rows={len(v1_alternatives_df)}")

    miss_after = int(v1["seqs"].eq("").sum())
    print(
        f"[saved] {V1_OOP} shape={v1.shape} | "
        f"missing seq before={len(missing_ids)} after={miss_after}"
    )
else:
    print(f"[skip] {RAW_V1} not found")




In [ ]:
# 7) Build dinucleotide properties from website sequence data (RNA alphabet A,C,G,U)
# Keep website (V2) separate from V1: do not read V1_OOP/RAW_V1 here.
seq_sources = [WEBSITE_SEQS, WEBSITE_OOP, WEBSITE_FULL]
seqs = []
for src in seq_sources:
    if not src.exists():
        continue
    s = pd.read_csv(src, dtype=str)
    s.columns = [c.lower() for c in s.columns]
    col = "seqs" if "seqs" in s.columns else ("seq" if "seq" in s.columns else None)
    if not col:
        continue
    seqs = s[col].dropna().map(normalize_sequence_value)
    seqs = [seq for seq in seqs.tolist() if has_usable_sequence_value(seq)]
    print(f"Using sequences from {src} ({len(seqs)} entries)")
    break

if seqs:
    from collections import Counter

    counts = Counter()
    total = 0
    for seq in seqs:
        x = "".join(seq.split()).upper().replace("T", "U")
        for i in range(len(x) - 1):
            d = x[i:i+2]
            if len(d) == 2 and set(d) <= set("ACGU"):
                counts[d] += 1
                total += 1

    rows = []
    for d in [a + b for a in "ACGU" for b in "ACGU"]:
        c = counts[d]
        rows.append({"dinuc": d, "count": c, "freq": (c / total if total else 0.0)})

    props = pd.DataFrame(rows)
    props.to_csv(DINUC_PROPS, index=False)
    print(f"[saved] {DINUC_PROPS} shape={props.shape}")
else:
    print("[skip] no website sequence source found for dinuc props")


In [ ]:
# 8) Build Disease Ontology helpers (no external parser required)
# This parser reads HumanDO.obo directly and keeps all synonym strings.

import re
import json


def extract_quoted_synonym(line: str):
    """Extract first quoted synonym string from an OBO synonym line."""
    q1 = line.find('"')
    if q1 == -1:
        return None

    buf = []
    escape = False
    for ch in line[q1 + 1:]:
        if escape:
            buf.append(ch)
            escape = False
            continue
        if ch == '\\':
            escape = True
            continue
        if ch == '"':
            return ''.join(buf)
        buf.append(ch)
    return None


def parse_do_obo(obo_path: Path):
    terms = []
    edges = []
    current = None

    def flush_term(t):
        if not t or not t.get('id'):
            return

        is_obsolete = str(t.get('is_obsolete', 'false')).lower() == 'true'
        if is_obsolete:
            return

        syns = [str(x).strip() for x in t.get('synonyms', []) if str(x).strip()]
        fallback_syn = str(t.get('name', '')).strip() or str(t.get('id', '')).strip()
        if not syns and fallback_syn:
            syns = [fallback_syn]

        terms.append({
            'doid': t.get('id', ''),
            'name': t.get('name', ''),
            'synonyms': ' || '.join(syns),
            'synonyms_json': json.dumps(syns, ensure_ascii=False),
            'synonym_count': len(syns),
        })

        for parent in t.get('is_a', []):
            edges.append({'child': t['id'], 'parent': parent})

    with obo_path.open('r', encoding='utf-8') as fh:
        for raw in fh:
            line = raw.strip()

            if not line:
                continue

            if line == '[Term]':
                flush_term(current)
                current = {'synonyms': [], 'is_a': []}
                continue

            if line.startswith('[') and line != '[Term]':
                flush_term(current)
                current = None
                continue

            if current is None:
                continue

            if line.startswith('id: '):
                current['id'] = line[4:].strip()
            elif line.startswith('name: '):
                current['name'] = line[6:].strip()
            elif line.startswith('is_obsolete: '):
                current['is_obsolete'] = line.split(':', 1)[1].strip()
            elif line.startswith('is_a: '):
                parent = line[6:].split(' ! ', 1)[0].strip()
                if parent:
                    current['is_a'].append(parent)
            elif line.startswith('synonym: '):
                txt = extract_quoted_synonym(line)
                if txt is not None:
                    current['synonyms'].append(txt)

    flush_term(current)

    terms_df = pd.DataFrame(terms).drop_duplicates(subset=['doid'])
    edges_df = pd.DataFrame(edges).drop_duplicates()
    return terms_df, edges_df


if DO_OBO.exists():
    terms_df, edges_df = parse_do_obo(DO_OBO)
    terms_df.to_csv(DO_TERMS, index=False)
    edges_df.to_csv(DO_EDGES, index=False)
    print(f"[saved] {DO_TERMS} shape={terms_df.shape}")
    print(f"[saved] {DO_EDGES} shape={edges_df.shape}")

    diseases = [c for c in disease_mat.columns if c != 'ID']

    def norm_text(s: str) -> str:
        s = str(s or '').strip().lower()
        s = re.sub(r'[^a-z0-9\s]', ' ', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s

    t = terms_df.fillna('').copy()
    t['name_norm'] = t['name'].map(norm_text)

    def parse_syns(row):
        if row.get('synonyms_json', ''):
            try:
                vals = json.loads(row['synonyms_json'])
                return [norm_text(x) for x in vals if norm_text(x)]
            except Exception:
                pass
        raw = str(row.get('synonyms', ''))
        return [norm_text(x) for x in raw.split(' || ') if norm_text(x)]

    t['syn_norms'] = t.apply(parse_syns, axis=1)

    name_to_doid, syn_to_doid = {}, {}
    for _, row in t.iterrows():
        doid = row['doid']
        nn = row['name_norm']
        if nn and nn not in name_to_doid:
            name_to_doid[nn] = doid
        for sn in row['syn_norms']:
            if sn and sn not in syn_to_doid:
                syn_to_doid[sn] = doid

    mapping = []
    for d in diseases:
        dn = norm_text(d)
        term = name_to_doid.get(dn, '') or syn_to_doid.get(dn, '')
        mapping.append({'disease': d, 'term': term})

    map_df = pd.DataFrame(mapping)
    map_df.to_csv(DO_MAP, index=False)
    mapped_n = int((map_df['term'] != '').sum())
    print(f"[saved] {DO_MAP} shape={map_df.shape} (mapped={mapped_n}/{len(map_df)})")

    if mapped_n < len(map_df):
        unmatched = map_df.loc[map_df['term'] == '', 'disease'].head(20).tolist()
        print('sample unmatched diseases:', unmatched)
else:
    print(f"[skip] {DO_OBO} not found")


In [ ]:
# 9) Validation summary
for p in [
    SYMBOLS_TXT,
    WEBSITE_SEQS,
    FETCH_REPORT,
    WEBSITE_MISSING_IDS,
    UNRESOLVED_IDS,
    AMBIGUOUS_ALTERNATIVES,
    WEBSITE_DISEASE,
    WEBSITE_FULL,
    WEBSITE_OOP,
    V1_OOP,
    V1_FETCH_REPORT,
    V1_UNRESOLVED_IDS,
    V1_AMBIGUOUS_ALTERNATIVES,
    DINUC_PROPS,
    DO_TERMS,
    DO_EDGES,
    DO_MAP,
]:
    if p.exists():
        try:
            d = pd.read_csv(p)
            print(f"{p}: shape={d.shape}")
        except Exception:
            print(f"{p}: exists (non-tabular)")
    else:
        print(f"{p}: missing")

if WEBSITE_SEQS.exists():
    s = pd.read_csv(WEBSITE_SEQS)
    if "seqs" in s.columns:
        print("website_sequences missing seq count:", int((~s["seqs"].map(normalize_sequence_value).map(has_usable_sequence_value)).sum()))
        print("website_sequences unique IDs:", s["ID"].nunique() if "ID" in s.columns else "n/a")

if WEBSITE_FULL.exists() and WEBSITE_DISEASE.exists():
    f = pd.read_csv(WEBSITE_FULL)
    y = pd.read_csv(WEBSITE_DISEASE)
    print("full matrix has seqs col:", "seqs" in f.columns)
    print("disease matrix unique IDs:", y["ID"].nunique() if "ID" in y.columns else "n/a")
    print("full matrix unique IDs:", f["ID"].nunique() if "ID" in f.columns else "n/a")
    if "seqs" in f.columns:
        print("full matrix missing seq count:", int((~f["seqs"].map(normalize_sequence_value).map(has_usable_sequence_value)).sum()))


